In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.4/398.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.2 MB/s eta 0:00:00
  Attempting uninstall: cython
    Found existing installation: Cython 3.0.12
    Uninstalling Cython-3.0.12:
      Successfully uninstalled Cython-3.0.12
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
xarray 2025.3.1 requires pandas

In [1]:
!pip install --upgrade --force-reinstall numpy pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.2
    Uninstalling tzdata-2025.2:
      Successfully uninstalled tzdata-2025.2
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1

In [3]:
# Ensure dependencies are installed BEFORE importing
#!pip install --upgrade --force-reinstall numpy pandas
#!pip install -r requirements.txt
#!pip install dice-ml # Explicitly install dice-ml

# It's often necessary to restart the runtime after significant package changes,
# especially with numpy/pandas. You can do this manually or add code like:
# import os
# os.kill(os.getpid(), 9)
# (Note: This will stop execution, requiring you to run cells again from the start)

from IPython import get_ipython
from IPython.display import display
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import dice_ml
from dice_ml import Dice
import joblib


# Mount Google Drive
drive.mount('/content/drive')


# Load dataset (first 5000 rows for quick experimentation)
data = pd.read_csv('/content/drive/MyDrive/DS Project/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')

# Remove commas and convert to numeric where needed
for col in ["Length of Stay", "Birth Weight", "Total Charges", "Total Costs"]:
    data[col] = pd.to_numeric(data[col].astype(str).str.replace(',', ''), errors='coerce')

# Fill missing numeric values with median
for col in data.columns:
    if data[col].dtype != "object":
        data[col] = data[col].fillna(data[col].median())

# Define categorical columns (keep as categorical dtype, no one-hot encoding)
categorical_columns = [
    'Hospital Service Area', 'Hospital County', 'Facility Name', 'Age Group',
    'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Type of Admission',
    'Patient Disposition', 'CCSR Diagnosis Code', 'CCSR Diagnosis Description',
    'CCSR Procedure Code', 'CCSR Procedure Description', 'APR DRG Description',
    'APR MDC Description', 'APR Severity of Illness Description',
    'APR Risk of Mortality', 'APR Medical Surgical Description',
    'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3',
    'Birth Weight', # Birth Weight is likely numeric, but was in the list, keeping for consistency with original code
    'Emergency Department Indicator'
]

# Apply category dtype BEFORE splitting
# This ensures both train and test sets have the same categories and handle unknown ones
for col in categorical_columns:
    if col in data.columns: # Check if column exists
        data[col] = data[col].astype('category')

# Prepare features and target variable
X = data.drop(columns=['Total Costs', 'Total Charges'])
y = data['Total Costs']

# Split into train and test sets
# By splitting after setting categories, X_test will inherit categories from X_train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing numeric values in train and test with train median
for col in X_train.columns:
    if X_train[col].dtype != 'category':  # numeric columns
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

# Impute missing categorical values with 'Missing' category (add if needed)
# Missing values resulting from train_test_split (due to categories not in train)
# will be handled here.
for col in categorical_columns:
     if col in X_train.columns: # Check if column exists
         # Add 'Missing' category if it doesn't exist
         if 'Missing' not in X_train[col].cat.categories:
             X_train[col] = X_train[col].cat.add_categories(['Missing'])
         if 'Missing' not in X_test[col].cat.categories:
              X_test[col] = X_test[col].cat.add_categories(['Missing'])

         X_train[col] = X_train[col].fillna('Missing')
         X_test[col] = X_test[col].fillna('Missing')


# Clean column names in X, X_train, X_test to remove problematic chars
def clean_col_names(cols):
    cleaned = []
    for col in cols:
        c = col
        if "[" in c or "]" in c:
            c = c.replace('[', '').replace(']', '')
        if "<" in c:
            c = c.replace('<', 'less than')
        cleaned.append(c)
    return cleaned

# Apply cleaning to all relevant dataframes
X.columns = clean_col_names(X.columns)
X_train.columns = clean_col_names(X_train.columns)
X_test.columns = clean_col_names(X_test.columns)


# Prepare full DataFrame for DiCE (features + target)
# Ensure column order is consistent between X and y before concat
X = X[X_train.columns] # Reindex X to match X_train column order
df_full = pd.concat([X, y], axis=1)

# DiCE expects numeric features as continuous_features, categorical features separately
# Filter categorical columns to only include those actually present in the data
actual_categorical_columns = [col for col in categorical_columns if col in df_full.columns]

data_reg = dice_ml.Data(
    dataframe=df_full,
    continuous_features=[col for col in df_full.columns if col not in actual_categorical_columns and col != 'Total Costs'],
    categorical_features=actual_categorical_columns,
    outcome_name='Total Costs'
)

# Load your pre-trained model (trained on the same data format)
# Ensure the model was trained on data with the same column names and dtypes
model = joblib.load("/content/drive/MyDrive/DS Project/lgb_model.pkl")

# Wrap the model for DiCE
model_dice_reg = dice_ml.Model(model=model, backend="sklearn", model_type='regressor')

# Initialize DiCE explainer
exp_reg = Dice(data_reg, model_dice_reg)

# Select a query instance (make sure to clean columns too)
query_instance_reg = X_test.iloc[[0]].copy()
# Column cleaning is already applied to X_test above, so this line is redundant here
# query_instance_reg.columns = clean_col_names(query_instance_reg.columns)

# Ensure the query instance has the exact same columns and order as the training data DiCE saw
query_instance_reg = query_instance_reg[X_train.columns]

# Define safe features to vary (exclude problematic categorical ones)
safe_features_to_vary = [col for col in X.columns if col not in [
    'CCSR Procedure Code', 'CCSR Procedure Description',  # problematic codes
    'Facility Name',  # high-cardinality, likely causes mismatches
    'Payment Typology 3'  # previously caused issues
]]

# Generate counterfactuals with restricted features
cf_reg = exp_reg.generate_counterfactuals(
    query_instance_reg,
    total_CFs=3,
    desired_range=[2000, 5000],
    features_to_vary=safe_features_to_vary
)


# Visualize and print counterfactuals
cf_reg.visualize_as_dataframe()
print(cf_reg.cf_examples_list[0].final_cfs_df)

# Save counterfactual explanations to a text file
with open("counterfactual_explanation.txt", "w") as f:
    f.write(cf_reg.cf_examples_list[0].final_cfs_df.to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<ipython-input-3-977fc708bf61>:28: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/content/drive/MyDrive/DS Project/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')
  0%|          | 0/1 [00:00<?, ?it/s]


ValueError: ('Feature', 'CCSR Procedure Code', 'has a value outside the dataset.')